In [1]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier, Pool
import torch
from itertools import product

In [2]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    SAVE_OOF = True
    SAVE_PRED = True
    SAVE_SUB = True

    MODEL_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.0,
        "grow_policy": "SymmetricTree",
        "bootstrap_type": "Bayesian",
        "random_seed": SEED,
        "verbose": 200,
        "early_stopping_rounds": 200,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
    }

In [3]:
# =========================
# Column groups
# =========================
NUM_COLS_BASE = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

CAT_COLS_BASE = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

BIN_COLS_BASE = [
    "Mulching_Used",
]

CAT_ALL_BASE = CAT_COLS_BASE + BIN_COLS_BASE
FEATURE_COLS_BASE = NUM_COLS_BASE + CAT_COLS_BASE + BIN_COLS_BASE

In [4]:
# =========================
# Utilities
# =========================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def load_data():
    train_df = pd.read_csv(CFG.TRAIN_PATH)
    test_df = pd.read_csv(CFG.TEST_PATH)
    orig_df = pd.read_csv(CFG.ORIG_PATH)
    return train_df, test_df, orig_df


def validate_columns(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    expected_train = set(FEATURE_COLS_BASE + [CFG.TARGET_COL, CFG.ID_COL])
    expected_test = set(FEATURE_COLS_BASE + [CFG.ID_COL])

    train_missing = expected_train - set(train_df.columns)
    train_extra = set(train_df.columns) - expected_train

    test_missing = expected_test - set(test_df.columns)
    test_extra = set(test_df.columns) - expected_test

    print("train_missing:", train_missing)
    print("train_extra:", train_extra)
    print("test_missing:", test_missing)
    print("test_extra:", test_extra)

    if train_missing or test_missing:
        raise ValueError("Column mismatch detected.")


def cast_dtypes(train_df: pd.DataFrame, test_df: pd.DataFrame, orig_df: pd.DataFrame):
    for col in CAT_ALL_BASE:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)
        orig_df[col] = orig_df[col].astype(str)

    for col in NUM_COLS_BASE:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
        orig_df[col] = pd.to_numeric(orig_df[col], errors="coerce")

    return train_df, test_df, orig_df


def make_pools(X_tr, y_tr, X_va, y_va, X_te):
    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_te,
        cat_features=CAT_ALL_BASE,
    )
    return train_pool, valid_pool, test_pool


def predict_with_bias(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    adjusted = proba + bias.reshape(1, -1)
    return np.argmax(adjusted, axis=1)


def evaluate_per_class_recall(y_true: np.ndarray, y_pred: np.ndarray, class_names=None):
    n_classes = len(np.unique(y_true))
    recalls = {}
    for cls_idx in range(n_classes):
        mask = (y_true == cls_idx)
        recall = float((y_pred[mask] == y_true[mask]).mean()) if mask.sum() > 0 else np.nan
        key = class_names[cls_idx] if class_names is not None else cls_idx
        recalls[key] = recall
    return recalls


def search_best_class_bias(
    oof_proba: np.ndarray,
    y_true: np.ndarray,
    bias_grid=None,
    fix_first_class_zero=True,
    verbose=True,
):
    """
    oof_proba: (n_samples, n_classes)
    y_true: encoded labels
    bias_grid: 例 [-0.30, -0.20, -0.10, -0.05, 0.0, 0.05, 0.10, 0.20, 0.30]
    fix_first_class_zero=True なら、bias[0] を 0 に固定して探索空間を減らす
    """

    n_classes = oof_proba.shape[1]

    if bias_grid is None:
        bias_grid = [-0.30, -0.20, -0.10, -0.05, 0.0, 0.05, 0.10, 0.20, 0.30]

    best_score = -1.0
    best_bias = np.zeros(n_classes, dtype=np.float32)

    if fix_first_class_zero:
        for rest in product(bias_grid, repeat=n_classes - 1):
            bias = np.array([0.0] + list(rest), dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)

            if score > best_score:
                best_score = score
                best_bias = bias.copy()
    else:
        for bias_tuple in product(bias_grid, repeat=n_classes):
            bias = np.array(bias_tuple, dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)

            if score > best_score:
                best_score = score
                best_bias = bias.copy()

    if verbose:
        print("best_bias:", best_bias)
        print(f"best_bias_score: {best_score:.6f}")

    return best_bias, best_score


def refine_class_bias(
    oof_proba: np.ndarray,
    y_true: np.ndarray,
    coarse_bias: np.ndarray,
    step=0.02,
    radius=0.08,
    fix_first_class_zero=True,
    verbose=True,
):
    """
    粗探索の近傍を細かく探索する
    """
    n_classes = oof_proba.shape[1]

    grids = []
    for i in range(n_classes):
        if fix_first_class_zero and i == 0:
            grids.append([0.0])
        else:
            center = coarse_bias[i]
            vals = np.arange(center - radius, center + radius + 1e-12, step)
            vals = np.round(vals, 6)
            grids.append(vals.tolist())

    best_score = -1.0
    best_bias = coarse_bias.copy()

    for bias_tuple in product(*grids):
        bias = np.array(bias_tuple, dtype=np.float32)
        pred = predict_with_bias(oof_proba, bias)
        score = balanced_accuracy_score(y_true, pred)

        if score > best_score:
            best_score = score
            best_bias = bias.copy()

    if verbose:
        print("refined_bias:", best_bias)
        print(f"refined_bias_score: {best_score:.6f}")

    return best_bias, best_score


ORIG_PRIOR_SOURCE_COLS = [
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
    "Mulching_Used",
]


def add_original_priors(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    target_col: str,
    label_encoder: LabelEncoder,
    source_cols=None,
    alpha: float = 20.0,
):
    if source_cols is None:
        source_cols = ORIG_PRIOR_SOURCE_COLS

    orig_y = label_encoder.transform(orig_df[target_col])
    n_classes = len(label_encoder.classes_)

    global_prior = np.bincount(orig_y, minlength=n_classes).astype(np.float64)
    global_prior = global_prior / global_prior.sum()

    created_cols = []

    for col in source_cols:
        tmp = orig_df[[col]].copy()
        tmp["_y"] = orig_y

        cnt = tmp.groupby(col).size().rename("cnt")

        class_cnt = (
            tmp.groupby([col, "_y"])
               .size()
               .unstack(fill_value=0)
        )

        for c in range(n_classes):
            if c not in class_cnt.columns:
                class_cnt[c] = 0
        class_cnt = class_cnt.sort_index(axis=1)

        probs = class_cnt.copy().astype(np.float64)
        for c in range(n_classes):
            probs[c] = (probs[c] + alpha * global_prior[c]) / (cnt + alpha)

        probs.columns = [f"orig_prior__{col}__{label_encoder.classes_[c]}" for c in range(n_classes)]
        probs = probs.reset_index()

        train_df = train_df.merge(probs, on=col, how="left")
        test_df = test_df.merge(probs, on=col, how="left")

        new_cols = [c for c in probs.columns if c != col]
        created_cols.extend(new_cols)

        for i, new_col in enumerate(new_cols):
            fill_value = global_prior[i]
            train_df[new_col] = train_df[new_col].fillna(fill_value)
            test_df[new_col] = test_df[new_col].fillna(fill_value)

    return train_df, test_df, created_cols

In [5]:
seed_everything(CFG.SEED)

train_df, test_df, orig_df = load_data()
train_df, test_df, orig_df = cast_dtypes(train_df, test_df, orig_df)

y_raw = train_df[CFG.TARGET_COL].copy()
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = list(le.classes_)
n_classes = len(class_names)

train_df, test_df, orig_prior_cols = add_original_priors(
    train_df=train_df,
    test_df=test_df,
    orig_df=orig_df,
    target_col=CFG.TARGET_COL,
    label_encoder=le,
    source_cols=ORIG_PRIOR_SOURCE_COLS,
    alpha=20.0,
)

FEATURE_COLS_ORIG_PRIORS = FEATURE_COLS_BASE + orig_prior_cols

print("classes:", class_names)
print("orig prior source cols:", ORIG_PRIOR_SOURCE_COLS)
print("n orig prior cols:", len(orig_prior_cols))
print("orig prior cols example:", orig_prior_cols[:10])

X = train_df[FEATURE_COLS_ORIG_PRIORS].copy()
X_test = test_df[FEATURE_COLS_ORIG_PRIORS].copy()

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("=" * 60)
    print(f"fold {fold} / {CFG.N_SPLITS}")

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_test,
        cat_features=CAT_ALL_BASE,
    )

    model = CatBoostClassifier(**CFG.MODEL_PARAMS)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    va_proba = model.predict_proba(valid_pool)
    te_proba = model.predict_proba(test_pool)

    oof_proba[va_idx] = va_proba
    test_proba += te_proba / CFG.N_SPLITS

    va_pred = np.argmax(va_proba, axis=1)
    fold_bacc = balanced_accuracy_score(y_va, va_pred)
    fold_scores.append(fold_bacc)

    print(f"fold {fold} balanced_accuracy = {fold_bacc:.6f}")

print("=" * 60)
print("fold scores:", [round(s, 6) for s in fold_scores])
print(f"cv mean balanced_accuracy = {np.mean(fold_scores):.6f}")
print(f"cv std  balanced_accuracy = {np.std(fold_scores):.6f}")

oof_pred = np.argmax(oof_proba, axis=1)
oof_bacc = balanced_accuracy_score(y, oof_pred)
print(f"OOF balanced_accuracy = {oof_bacc:.6f}")

print("\nPer-class recall on OOF:")
for cls_idx, cls_name in enumerate(class_names):
    mask = (y == cls_idx)
    recall = (oof_pred[mask] == y[mask]).mean()
    print(f"{cls_name}: {recall:.6f}")

test_pred = np.argmax(test_proba, axis=1)
test_label = le.inverse_transform(test_pred)

sub_df = pd.DataFrame({
    CFG.ID_COL: test_df[CFG.ID_COL],
    CFG.TARGET_COL: test_label
})

np.save("oof_catboost_orig_priors_sel_proba.npy", oof_proba)
np.save("pred_catboost_orig_priors_sel_proba.npy", test_proba)
sub_df.to_csv("submission_catboost_orig_priors_sel.csv", index=False)

print("\nSaved:")
print("- oof_catboost_orig_priors_sel_proba.npy")
print("- pred_catboost_orig_priors_sel_proba.npy")
print("- submission_catboost_orig_priors_sel.csv")

classes: ['High', 'Low', 'Medium']
orig prior source cols: ['Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Region', 'Mulching_Used']
n orig prior cols: 18
orig prior cols example: ['orig_prior__Crop_Growth_Stage__High', 'orig_prior__Crop_Growth_Stage__Low', 'orig_prior__Crop_Growth_Stage__Medium', 'orig_prior__Season__High', 'orig_prior__Season__Low', 'orig_prior__Season__Medium', 'orig_prior__Irrigation_Type__High', 'orig_prior__Irrigation_Type__Low', 'orig_prior__Irrigation_Type__Medium', 'orig_prior__Water_Source__High']
fold 1 / 5
0:	learn: 1.0062988	test: 1.0062700	best: 1.0062700 (0)	total: 130ms	remaining: 6m 31s
200:	learn: 0.0643030	test: 0.0645802	best: 0.0645802 (200)	total: 2.78s	remaining: 38.8s
400:	learn: 0.0612993	test: 0.0626332	best: 0.0626332 (400)	total: 5.28s	remaining: 34.2s
600:	learn: 0.0593400	test: 0.0616746	best: 0.0616732 (599)	total: 7.78s	remaining: 31s
800:	learn: 0.0581005	test: 0.0611982	best: 0.0611981 (799)	total: 10.1s	remaining: 